In [1]:
from pyarrow.lib import tobytes


In [ ]:
import pyarrow as pa
from pyarrow.lib import tobytes
import pyarrow.substrait as substrait
test_table_1 = pa.Table.from_pydict({"x": [1, 2, 3]})
test_table_2 = pa.Table.from_pydict({"x": [4, 5, 6]})
def table_provider(names, schema):
    if not names:
       raise Exception("No names provided")
    elif names[0] == "t1":
       return test_table_1
    elif names[1] == "t2":
       return test_table_2
    else:
       raise Exception("Unrecognized table name")

substrait_query = '''
        {
            "relations": [
            {"rel": {
                "read": {
                "base_schema": {
                    "struct": {
                    "types": [
                                {"i64": {}}
                            ]
                    },
                    "names": [
                            "x"
                            ]
                },
                "namedTable": {
                        "names": ["t1"]
                }
                }
            }}
            ]
        }
'''
buf = pa._substrait._parse_json_plan(tobytes(substrait_query))
reader = pa.substrait.run_query(buf, table_provider=table_provider)
reader.read_all()

<class 'function'>


pyarrow.Table
x: int64
----
x: [[1,2,3]]

In [4]:
import random
import json

n = 1000

# status[i] is either 0 or 1
status = [random.randint(0, 1) for _ in range(n)]

# candies[i] is between 1 and 1000
candies = [random.randint(1, 1000) for _ in range(n)]

# keys[i] is a unique list of box indices that box i can open
keys = []
for i in range(n):
    num_keys = random.randint(0, min(10, n-1))
    key_set = random.sample([j for j in range(n) if j != i], num_keys)
    keys.append(key_set)

# containedBoxes[i] is a list of unique boxes found inside box i
containedBoxes = []
for i in range(n):
    num_contained = random.randint(0, min(10, n-1))
    contained_set = random.sample([j for j in range(n) if j != i], num_contained)
    containedBoxes.append(contained_set)

# initialBoxes is a list of up to n starting boxes
initialBoxes = random.sample(range(n), random.randint(1, n))

# Print in JSON format
test_case = {
    "status": status,
    "candies": candies,
    "keys": keys,
    "containedBoxes": containedBoxes,
    "initialBoxes": initialBoxes
}

for key in test_case:
    print(f"{key}: {test_case[key]}")

status: [0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 0, 0, 0, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 0, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 0, 0, 1, 0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 0, 1, 1, 1, 0, 1, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0, 0, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1

In [5]:
import random
ids = list(range(1, random.randint(1, 100)))
names = [f"Employee{i}" for i in ids]
salaries = [20000 + (i % 20) * 1000 for i in ids]
projects = [f"Project{i}" for i in ids]
projids = [i for i in ids]
employees_schema = pa.schema([
    ("id", pa.int32()),
    ("name", pa.string()),
    ("salary", pa.int32())
])

project_schema = pa.schema([
    ("id", pa.int32()),
    ("projname", pa.string()),
    ("projid", pa.int32())
])

employees_table = pa.table([
    pa.array(ids),
    pa.array(names),
    pa.array(salaries)
], schema=employees_schema)

projects_table = pa.table([
    pa.array(ids),
    pa.array(projects),
    pa.array(projids)
], schema=project_schema)

def table_provider(named_table, schema):
    return employees_table, projects_table

In [ ]:
subs = '''

                      }, {
                        "i32": {
                          "type_variation_reference": 0,
                          "nullability": "NULLABILITY_NULLABLE"
                        }
                      }],
                      "type_variation_reference": 0,
                      "nullability": "NULLABILITY_REQUIRED"
                    }
                  },
                  "named_table": {
                    "names": ["PROJECTS"]
                  }
                }
              },
              "expression": {
                "scalar_function": {
                  "function_reference": 0,
                  "args": [],
                  "output_type": {
                    "bool": {
                      "type_variation_reference": 0,
                      "nullability": "NULLABILITY_NULLABLE"
                    }
                  },
                  "arguments": [{
                    "value": {
                      "selection": {
                        "direct_reference": {
                          "struct_field": {
                            "field": 0
                          }
                        },
                        "root_reference": {
                        }
                      }
                    }
                  }, {
                    "value": {
                      "selection": {
                        "direct_reference": {
                          "struct_field": {
                            "field": 3
                          }
                        },
                        "root_reference": {
                        }
                      }
                    }
                  }],
                  "options": []
                }
              },
              "type": "JOIN_TYPE_INNER"
            }
          },
          "expressions": [{
            "selection": {
              "direct_reference": {
                "struct_field": {
                  "field": 2
                }
              },
              "root_reference": {
              }
            }
          }, {
            "selection": {
              "direct_reference": {
                "struct_field": {
                  "field": 5
                }
              },
              "root_reference": {
              }
            }
          }]
        }
      },
      "names": ["SALARY", "PROJID"]
    }
  }],
  "expected_type_urls": []
}

'''